# 01 — Génération du dataset synthétique (version enrichie)
### CHU Mohammed VI — Agadir | Module Hospitalisations

**Objectif de ce notebook :**
- Générer un jeu de données synthétique **réaliste et enrichi** : patients, services, médecins, hospitalisations
- Simuler des volumes proches d'un vrai CHU
- Injecter volontairement des anomalies (valeurs manquantes, doublons, dates incohérentes, âges invalides)
- Exporter les fichiers CSV bruts dans `data/raw/`

Ces fichiers "sales" serviront de point de départ au notebook `02_audit_qualite_donnees.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import timedelta
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('fr_FR')
Faker.seed(SEED)

print("Librairies chargées avec succès ✅")

Librairies chargées avec succès ✅


## 1. Paramètres de configuration

Volumes réglés pour simuler un vrai CHU. Tu peux repasser en `MODE_TEST = True` si tu veux itérer rapidement pendant le développement.

In [2]:
# --- Mode test (petit volume, rapide) vs mode complet (volume réaliste) ---
MODE_TEST = False

if MODE_TEST:
    N_PATIENTS = 500
    N_HOSPITALISATIONS = 800
else:
    N_PATIENTS = 8000
    N_HOSPITALISATIONS = 15000

OUTPUT_DIR = "../data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VILLES_MAROC = [
    "Agadir", "Casablanca", "Marrakech", "Rabat", "Tanger",
    "Fès", "Meknès", "Oujda", "Kénitra", "Tétouan",
    "Safi", "Taroudant", "Essaouira", "Inezgane", "Chtouka Aït Baha",
    "Tiznit", "Ait Melloul", "Guelmim", "Sidi Ifni", "Ouarzazate"
]

print(f"Mode : {'TEST' if MODE_TEST else 'COMPLET'}")
print(f"Configuration : {N_PATIENTS} patients / {N_HOSPITALISATIONS} hospitalisations")

Mode : COMPLET
Configuration : 8000 patients / 15000 hospitalisations


## 2. Table `services` (élargie, ~16 services comme dans un vrai CHU)

In [3]:
SERVICES = [
    {"nom_service": "Cardiologie",         "categorie": "Médecine",             "capacite_lits": 30, "etage": 2},
    {"nom_service": "Pédiatrie",           "categorie": "Médecine",             "capacite_lits": 25, "etage": 1},
    {"nom_service": "Chirurgie générale",  "categorie": "Chirurgie",            "capacite_lits": 35, "etage": 3},
    {"nom_service": "Urgences",            "categorie": "Urgences",             "capacite_lits": 20, "etage": 0},
    {"nom_service": "Maternité",           "categorie": "Gynéco-Obstétrique",   "capacite_lits": 40, "etage": 1},
    {"nom_service": "Oncologie",           "categorie": "Médecine",             "capacite_lits": 20, "etage": 4},
    {"nom_service": "Néphrologie",         "categorie": "Médecine",             "capacite_lits": 15, "etage": 2},
    {"nom_service": "Médecine interne",    "categorie": "Médecine",             "capacite_lits": 30, "etage": 2},
    {"nom_service": "Orthopédie",          "categorie": "Chirurgie",            "capacite_lits": 25, "etage": 3},
    {"nom_service": "Pneumologie",         "categorie": "Médecine",             "capacite_lits": 20, "etage": 2},
    {"nom_service": "Gastro-entérologie",  "categorie": "Médecine",             "capacite_lits": 18, "etage": 2},
    {"nom_service": "Urologie",            "categorie": "Chirurgie",            "capacite_lits": 20, "etage": 3},
    {"nom_service": "Neurologie",          "categorie": "Médecine",             "capacite_lits": 18, "etage": 4},
    {"nom_service": "Réanimation",         "categorie": "Soins Intensifs",      "capacite_lits": 15, "etage": 0},
    {"nom_service": "Dermatologie",        "categorie": "Médecine",             "capacite_lits": 12, "etage": 1},
    {"nom_service": "ORL",                 "categorie": "Chirurgie",            "capacite_lits": 15, "etage": 3},
]

df_services = pd.DataFrame(SERVICES)
df_services.insert(0, "id_service", range(1, len(df_services) + 1))

print(f"{len(df_services)} services créés")
df_services

16 services créés


,id_service,nom_service,categorie,capacite_lits,etage
0,1,Cardiologie,Médecine,30,2
1,2,Pédiatrie,Médecine,25,1
2,3,Chirurgie générale,Chirurgie,35,3
3,4,Urgences,Urgences,20,0
4,5,Maternité,Gynéco-Obstétrique,40,1
5,6,Oncologie,Médecine,20,4
6,7,Néphrologie,Médecine,15,2
7,8,Médecine interne,Médecine,30,2
8,9,Orthopédie,Chirurgie,25,3
9,10,Pneumologie,Médecine,20,2


## 3. Table `medecins` (nouvelle — rattachés à un service)

Chaque service a entre 3 et 8 médecins, avec une spécialité cohérente avec leur service et un nombre d'années d'expérience.

In [4]:
PRENOMS_MASCULINS_MAROC = [
    "Mohammed", "Ahmed", "Youssef", "Hamza", "Ayoub", "Karim", "Rachid", "Omar",
    "Yassine", "Amine", "Anas", "Ismail", "Khalid", "Reda", "Soufiane", "Adil",
    "Mehdi", "Nabil", "Tarik", "Zakaria", "Hassan", "Abdellah", "Said", "Driss",
    "Abderrahim", "Aziz", "Bilal", "Brahim", "Chakib", "Fouad", "Hamid", "Idriss",
    "Jamal", "Karam", "Lahcen", "Mounir", "Noureddine", "Othmane", "Rida", "Samir",
    "Taha", "Walid", "Yassir", "Ziad", "Abdelkader", "Abdelali", "Achraf", "Badr",
    "Chafik", "Elhassane", "Fahd", "Ghali", "Hicham", "Imad", "Jawad", "Kamal",
    "Marouane", "Nourdine", "Othman", "Rachad"
]

PRENOMS_FEMININS_MAROC = [
    "Fatima", "Khadija", "Salma", "Meryem", "Amina", "Nadia", "Hiba", "Imane",
    "Sara", "Zineb", "Hajar", "Loubna", "Ghita", "Kenza", "Sanae", "Naima",
    "Btissam", "Ilham", "Samira", "Asmaa", "Karima", "Latifa", "Wafaa", "Souad",
    "Aicha", "Bouchra", "Chaimae", "Dounia", "Fadwa", "Hanane", "Ibtissam", "Jihane",
    "Karima", "Laila", "Malak", "Nawal", "Ouafae", "Rania", "Siham", "Touria",
    "Widad", "Yasmine", "Zahra", "Amal", "Basma", "Chahd", "Doha", "Fatiha",
    "Ghizlane", "Houda", "Ikram", "Jamila", "Khaoula", "Leila", "Meryam", "Nisrine",
    "Oumaima", "Rachida", "Safaa", "Yousra"
]

NOMS_FAMILLE_MAROC = [
    "Alaoui", "Bennani", "El Idrissi", "Tazi", "Berrada", "Fassi", "El Amrani",
    "Chraibi", "Benjelloun", "Lahlou", "El Ouazzani", "Sbai", "Cherkaoui",
    "El Fassi", "Bennis", "Kabbaj", "Belhaj", "Ziani", "El Yousfi", "Guessous",
    "Idrissi", "Benkirane", "Ouali", "Sekkat", "Alami", "Benslimane", "Chaoui",
    "Dahbi", "El Fenni", "Ghazi", "Hakimi", "Jabri", "Kadiri", "Loukili", "Mansouri",
    "Naciri", "Ouahbi", "Qadiri", "Rifi", "Sahli", "Tahiri", "Zaki", "Amrani",
    "Bakkali", "Chentouf", "Doukkali", "El Gharbi", "Filali", "Habibi", "Jouahri",
    "Kettani"
]

# Spécialité du médecin = nom du service simplifié (cohérence médecin/service)
def generer_medecins_pour_service(id_service, nom_service, id_medecin_start):
    medecins = []
    nb_medecins = random.randint(3, 8)
    for i in range(nb_medecins):
        sexe = random.choice(["M", "F"])
        prenom = random.choice(PRENOMS_MASCULINS_MAROC) if sexe == "M" else random.choice(PRENOMS_FEMININS_MAROC)
        medecins.append({
            "id_medecin": id_medecin_start + i,
            "nom": random.choice(NOMS_FAMILLE_MAROC),
            "prenom": prenom,
            "sexe": sexe,
            "specialite": nom_service,
            "id_service": id_service,
            "annees_experience": random.randint(1, 35),
        })
    return medecins

medecins_list = []
id_medecin_courant = 1
for _, row in df_services.iterrows():
    medecins_service = generer_medecins_pour_service(row["id_service"], row["nom_service"], id_medecin_courant)
    medecins_list.extend(medecins_service)
    id_medecin_courant += len(medecins_service)

df_medecins = pd.DataFrame(medecins_list)

print(f"{len(df_medecins)} médecins créés répartis sur {len(df_services)} services")
df_medecins.head(10)

103 médecins créés répartis sur 16 services


,id_medecin,nom,prenom,sexe,specialite,id_service,annees_experience
0,1,Filali,Ahmed,M,Cardiologie,1,18
1,2,Benjelloun,Soufiane,M,Cardiologie,1,7
2,3,Dahbi,Othmane,M,Cardiologie,1,3
3,4,El Fassi,Karim,M,Cardiologie,1,15
4,5,Cherkaoui,Mounir,M,Cardiologie,1,35
5,6,El Fenni,Sanae,F,Cardiologie,1,18
6,7,El Ouazzani,Chafik,M,Cardiologie,1,28
7,8,Lahlou,Ilham,F,Cardiologie,1,14
8,9,Alami,Karim,M,Pédiatrie,2,7
9,10,Ouali,Meryam,F,Pédiatrie,2,17


## 4. Table `patients` (enrichie)

Ajout de champs réalistes : numéro de dossier, groupe sanguin, téléphone, type de couverture (assurance/mutuelle/CNSS/RAMED/sans couverture).

In [5]:
GROUPES_SANGUINS = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
# Distribution approximative réaliste (O+ et A+ plus fréquents)
POIDS_GROUPES_SANGUINS = [0.28, 0.05, 0.10, 0.02, 0.03, 0.01, 0.36, 0.06, 0.09][:len(GROUPES_SANGUINS)]
POIDS_GROUPES_SANGUINS = [0.34, 0.06, 0.09, 0.02, 0.03, 0.01, 0.38, 0.07]

COUVERTURES_SANTE = ["CNSS", "CNOPS", "RAMED", "Assurance privée", "Sans couverture"]
POIDS_COUVERTURES = [0.30, 0.15, 0.25, 0.10, 0.20]

def generer_patient(id_patient):
    sexe = random.choice(["M", "F"])
    prenom = random.choice(PRENOMS_MASCULINS_MAROC) if sexe == "M" else random.choice(PRENOMS_FEMININS_MAROC)
    date_naissance = fake.date_of_birth(minimum_age=0, maximum_age=95)
    age = (pd.Timestamp.today().date() - date_naissance).days // 365

    return {
        "id_patient": id_patient,
        "num_dossier": f"DOS-{100000 + id_patient}",
        "nom": random.choice(NOMS_FAMILLE_MAROC),
        "prenom": prenom,
        "date_naissance": date_naissance,
        "age": age,
        "sexe": sexe,
        "ville": random.choice(VILLES_MAROC),
        "telephone": fake.phone_number(),
        "groupe_sanguin": random.choices(GROUPES_SANGUINS, weights=POIDS_GROUPES_SANGUINS, k=1)[0],
        "couverture_sante": random.choices(COUVERTURES_SANTE, weights=POIDS_COUVERTURES, k=1)[0],
    }

patients_list = [generer_patient(i) for i in range(1, N_PATIENTS + 1)]
df_patients = pd.DataFrame(patients_list)

print(f"{len(df_patients)} patients créés")
df_patients.head(10)

8000 patients créés


,id_patient,num_dossier,nom,prenom,date_naissance,age,sexe,ville,telephone,groupe_sanguin,couverture_sante
0,1,DOS-100001,Benkirane,Badr,2017-07-31,8,M,Inezgane,+33 (0)1 49 43 32 18,O+,Sans couverture
1,2,DOS-100002,El Ouazzani,Nabil,1942-05-21,84,M,Rabat,01 58 01 33 89,A-,CNSS
2,3,DOS-100003,Cherkaoui,Sanae,1934-03-02,92,F,Chtouka Aït Baha,+33 5 61 37 94 02,A-,Sans couverture
3,4,DOS-100004,Bennani,Soufiane,2025-07-27,0,M,Meknès,04 83 42 35 11,A-,CNSS
4,5,DOS-100005,Ziani,Elhassane,1982-04-06,44,M,Taroudant,+33 (0)4 92 59 40 78,O+,CNOPS
5,6,DOS-100006,Chraibi,Khadija,1947-07-18,79,F,Kénitra,02 28 84 95 93,A+,Sans couverture
6,7,DOS-100007,El Amrani,Salma,2026-06-12,0,F,Ouarzazate,+33 (0)1 74 34 13 16,B+,Assurance privée
7,8,DOS-100008,Rifi,Dounia,1968-05-21,58,F,Ait Melloul,04 95 25 53 41,A+,Sans couverture
8,9,DOS-100009,El Idrissi,Mehdi,2013-06-16,13,M,Inezgane,+33 3 82 27 64 83,A+,Sans couverture
9,10,DOS-100010,Dahbi,Driss,2023-09-24,2,M,Marrakech,0179305641,O-,Sans couverture


## 5. Pathologies réalistes par service

Chaque service a ses pathologies typiques — plus cohérent qu'une pathologie générique aléatoire, et ça enrichira les analyses (EDA) et le modèle ML (feature `pathologie`).

In [6]:
PATHOLOGIES_PAR_SERVICE = {
    "Cardiologie": ["Infarctus du myocarde", "Insuffisance cardiaque", "Arythmie", "Hypertension artérielle sévère", "Angine de poitrine"],
    "Pédiatrie": ["Bronchiolite", "Gastro-entérite aiguë", "Pneumonie infantile", "Fièvre typhoïde", "Déshydratation sévère"],
    "Chirurgie générale": ["Appendicite", "Hernie inguinale", "Occlusion intestinale", "Cholécystite", "Péritonite"],
    "Urgences": ["Traumatisme crânien", "Polytraumatisme", "Intoxication aiguë", "Douleur abdominale aiguë", "AVC suspecté"],
    "Maternité": ["Accouchement normal", "Césarienne", "Pré-éclampsie", "Grossesse à risque", "Hémorragie post-partum"],
    "Oncologie": ["Cancer du sein", "Cancer colorectal", "Lymphome", "Cancer du poumon", "Leucémie"],
    "Néphrologie": ["Insuffisance rénale chronique", "Insuffisance rénale aiguë", "Lithiase rénale", "Néphrite", "Dialyse"],
    "Médecine interne": ["Diabète décompensé", "Anémie sévère", "Infection généralisée", "Bilan diagnostique complexe", "Fièvre inexpliquée"],
    "Orthopédie": ["Fracture du fémur", "Fracture du bras", "Prothèse de hanche", "Entorse grave", "Rupture ligamentaire"],
    "Pneumologie": ["Pneumonie", "Asthme sévère", "BPCO décompensée", "Tuberculose", "Embolie pulmonaire"],
    "Gastro-entérologie": ["Ulcère gastroduodénal", "Hépatite", "Pancréatite aiguë", "Colite", "Hémorragie digestive"],
    "Urologie": ["Colique néphrétique", "Infection urinaire sévère", "Hypertrophie prostatique", "Cancer de la vessie", "Rétention urinaire"],
    "Neurologie": ["AVC ischémique", "Épilepsie", "Migraine sévère", "Méningite", "Sclérose en plaques"],
    "Réanimation": ["Choc septique", "Détresse respiratoire aiguë", "Coma", "Polytraumatisme grave", "Défaillance multi-organes"],
    "Dermatologie": ["Brûlure étendue", "Infection cutanée sévère", "Psoriasis sévère", "Eczéma sévère", "Cellulite"],
    "ORL": ["Otite sévère", "Sinusite chronique", "Amygdalite", "Surdité brutale", "Vertige sévère"],
}

MODES_SORTIE = ["Guéri", "Amélioré", "Transfert vers un autre service", "Sortie contre avis médical", "Décès"]
POIDS_MODES_SORTIE = [0.55, 0.25, 0.10, 0.06, 0.04]

print("Pathologies définies pour", len(PATHOLOGIES_PAR_SERVICE), "services")

Pathologies définies pour 16 services


## 6. Table `hospitalisations` (enrichie)

Ajout de : `id_medecin` (rattaché au bon service), `pathologie` (cohérente avec le service), `mode_sortie`, `numero_chambre`.

In [7]:
# Regrouper les médecins par service pour un tirage cohérent
medecins_par_service = df_medecins.groupby("id_service")["id_medecin"].apply(list).to_dict()

# --- Durée de base par pathologie (en jours), reflète la sévérité clinique ---
DUREE_BASE_PATHOLOGIE = {
    # Cardiologie
    "Infarctus du myocarde": 9, "Insuffisance cardiaque": 8, "Arythmie": 4,
    "Hypertension artérielle sévère": 3, "Angine de poitrine": 3,
    # Pédiatrie
    "Bronchiolite": 4, "Gastro-entérite aiguë": 3, "Pneumonie infantile": 6,
    "Fièvre typhoïde": 7, "Déshydratation sévère": 3,
    # Chirurgie générale
    "Appendicite": 4, "Hernie inguinale": 3, "Occlusion intestinale": 7,
    "Cholécystite": 5, "Péritonite": 9,
    # Urgences
    "Traumatisme crânien": 5, "Polytraumatisme": 8, "Intoxication aiguë": 3,
    "Douleur abdominale aiguë": 2, "AVC suspecté": 6,
    # Maternité
    "Accouchement normal": 2, "Césarienne": 4, "Pré-éclampsie": 5,
    "Grossesse à risque": 6, "Hémorragie post-partum": 4,
    # Oncologie
    "Cancer du sein": 10, "Cancer colorectal": 11, "Lymphome": 9,
    "Cancer du poumon": 12, "Leucémie": 14,
    # Néphrologie
    "Insuffisance rénale chronique": 10, "Insuffisance rénale aiguë": 6,
    "Lithiase rénale": 3, "Néphrite": 5, "Dialyse": 2,
    # Médecine interne
    "Diabète décompensé": 5, "Anémie sévère": 4, "Infection généralisée": 7,
    "Bilan diagnostique complexe": 6, "Fièvre inexpliquée": 5,
    # Orthopédie
    "Fracture du fémur": 8, "Fracture du bras": 3, "Prothèse de hanche": 7,
    "Entorse grave": 2, "Rupture ligamentaire": 4,
    # Pneumologie
    "Pneumonie": 6, "Asthme sévère": 4, "BPCO décompensée": 8,
    "Tuberculose": 14, "Embolie pulmonaire": 7,
    # Gastro-entérologie
    "Ulcère gastroduodénal": 4, "Hépatite": 7, "Pancréatite aiguë": 8,
    "Colite": 5, "Hémorragie digestive": 6,
    # Urologie
    "Colique néphrétique": 2, "Infection urinaire sévère": 4,
    "Hypertrophie prostatique": 3, "Cancer de la vessie": 9, "Rétention urinaire": 2,
    # Neurologie
    "AVC ischémique": 9, "Épilepsie": 3, "Migraine sévère": 2,
    "Méningite": 8, "Sclérose en plaques": 6,
    # Réanimation
    "Choc septique": 12, "Détresse respiratoire aiguë": 10, "Coma": 15,
    "Polytraumatisme grave": 14, "Défaillance multi-organes": 16,
    # Dermatologie
    "Brûlure étendue": 10, "Infection cutanée sévère": 5, "Psoriasis sévère": 4,
    "Eczéma sévère": 3, "Cellulite": 4,
    # ORL
    "Otite sévère": 3, "Sinusite chronique": 3, "Amygdalite": 2,
    "Surdité brutale": 2, "Vertige sévère": 2,
}

def generer_hospitalisation(id_hospitalisation):
    id_patient = random.randint(1, N_PATIENTS)

    service_row = df_services.sample(1).iloc[0]
    id_service = int(service_row["id_service"])
    nom_service = service_row["nom_service"]
    capacite_lits = int(service_row["capacite_lits"])

    id_medecin = random.choice(medecins_par_service[id_service])
    pathologie = random.choice(PATHOLOGIES_PAR_SERVICE[nom_service])
    type_admission = random.choice(["Urgence", "Programmée", "Transfert"])

    date_entree = fake.date_between(start_date="-3y", end_date="today")

    # --- Durée de séjour dépendante de l'âge, la pathologie et le type d'admission ---
    age_patient = df_patients.loc[df_patients["id_patient"] == id_patient, "age"].values
    age_patient = age_patient[0] if len(age_patient) > 0 else 40

    duree_base = DUREE_BASE_PATHOLOGIE[pathologie]
    effet_age = max(0, (age_patient - 50) / 10)
    effet_admission = {"Urgence": 1.5, "Transfert": 2.0, "Programmée": 0}[type_admission]
    bruit_aleatoire = np.random.exponential(scale=1.5)

    duree_sejour = int(np.clip(duree_base + effet_age + effet_admission + bruit_aleatoire, 1, 60))
    date_sortie = date_entree + timedelta(days=duree_sejour)

    return {
        "id_hospitalisation": id_hospitalisation,
        "id_patient": id_patient,
        "id_service": id_service,
        "id_medecin": id_medecin,
        "date_entree": date_entree,
        "date_sortie": date_sortie,
        "type_admission": type_admission,
        "pathologie": pathologie,
        "mode_sortie": random.choices(MODES_SORTIE, weights=POIDS_MODES_SORTIE, k=1)[0],
        "numero_chambre": f"{random.randint(1, capacite_lits):03d}",
    }

hospitalisations_list = [generer_hospitalisation(i) for i in range(1, N_HOSPITALISATIONS + 1)]
df_hospitalisations = pd.DataFrame(hospitalisations_list)

print(f"{len(df_hospitalisations)} hospitalisations créées")
df_hospitalisations.head(10)

15000 hospitalisations créées


,id_hospitalisation,id_patient,id_service,id_medecin,date_entree,date_sortie,type_admission,pathologie,mode_sortie,numero_chambre
0,1,7070,1,3,2024-06-26,2024-07-06,Transfert,Insuffisance cardiaque,Transfert vers un autre service,017
1,2,1794,15,94,2024-03-03,2024-03-09,Transfert,Cellulite,Amélioré,009
2,3,4236,10,65,2023-12-18,2024-01-08,Programmée,Tuberculose,Transfert vers un autre service,018
3,4,4166,16,101,2024-01-10,2024-01-16,Programmée,Sinusite chronique,Amélioré,008
4,5,2853,1,7,2026-04-08,2026-04-16,Transfert,Angine de poitrine,Guéri,022
5,6,3530,3,15,2026-06-06,2026-06-11,Programmée,Cholécystite,Sortie contre avis médical,012
6,7,7691,6,35,2024-04-05,2024-04-20,Urgence,Cancer du sein,Amélioré,015
7,8,531,6,37,2024-09-24,2024-10-04,Programmée,Lymphome,Transfert vers un autre service,019
8,9,2908,7,40,2024-12-02,2024-12-13,Urgence,Insuffisance rénale chronique,Transfert vers un autre service,013
9,10,7710,6,35,2025-12-10,2025-12-23,Programmée,Cancer du poumon,Transfert vers un autre service,003


## 7. Injection volontaire d'anomalies

Toujours conforme au cahier des charges : valeurs manquantes, doublons, incohérences de dates, âges invalides. Les taux sont documentés pour le rapport qualité du notebook suivant.

In [8]:
TAUX_VALEURS_MANQUANTES = 0.04
TAUX_DOUBLONS = 0.02
TAUX_DATES_INCOHERENTES = 0.015
TAUX_AGES_INVALIDES = 0.01

rapport_anomalies = {}

In [9]:
# --- Valeurs manquantes dans hospitalisations (date_sortie, id_medecin, pathologie) ---
n_manquants = int(len(df_hospitalisations) * TAUX_VALEURS_MANQUANTES)
idx_manquants = np.random.choice(df_hospitalisations.index, size=n_manquants, replace=False)

tiers = n_manquants // 3
df_hospitalisations.loc[idx_manquants[:tiers], "date_sortie"] = np.nan
df_hospitalisations.loc[idx_manquants[tiers:2*tiers], "id_medecin"] = np.nan
df_hospitalisations.loc[idx_manquants[2*tiers:], "pathologie"] = np.nan

# --- Valeurs manquantes dans patients (âge, téléphone) ---
n_manquants_patients = int(len(df_patients) * TAUX_VALEURS_MANQUANTES)
idx_manquants_p = np.random.choice(df_patients.index, size=n_manquants_patients, replace=False)
moitie_p = n_manquants_patients // 2
df_patients.loc[idx_manquants_p[:moitie_p], "age"] = np.nan
df_patients.loc[idx_manquants_p[moitie_p:], "telephone"] = np.nan

rapport_anomalies["valeurs_manquantes_hospitalisations"] = n_manquants
rapport_anomalies["valeurs_manquantes_patients"] = n_manquants_patients

print(f"{n_manquants} valeurs manquantes injectées dans hospitalisations")
print(f"{n_manquants_patients} valeurs manquantes injectées dans patients")

600 valeurs manquantes injectées dans hospitalisations
320 valeurs manquantes injectées dans patients


In [10]:
# --- Doublons ---
n_doublons_hosp = int(len(df_hospitalisations) * TAUX_DOUBLONS)
lignes_a_dupliquer = df_hospitalisations.sample(n=n_doublons_hosp, random_state=SEED)
df_hospitalisations = pd.concat([df_hospitalisations, lignes_a_dupliquer], ignore_index=True)

n_doublons_patients = int(len(df_patients) * TAUX_DOUBLONS)
lignes_a_dupliquer_p = df_patients.sample(n=n_doublons_patients, random_state=SEED)
df_patients = pd.concat([df_patients, lignes_a_dupliquer_p], ignore_index=True)

rapport_anomalies["doublons_hospitalisations"] = n_doublons_hosp
rapport_anomalies["doublons_patients"] = n_doublons_patients

print(f"{n_doublons_hosp} doublons injectés dans hospitalisations")
print(f"{n_doublons_patients} doublons injectés dans patients")

300 doublons injectés dans hospitalisations
160 doublons injectés dans patients


In [11]:
# --- Dates incohérentes ---
mask_dates_valides = df_hospitalisations["date_sortie"].notna()
indices_valides = df_hospitalisations[mask_dates_valides].index

n_dates_incoherentes = int(len(df_hospitalisations) * TAUX_DATES_INCOHERENTES)
idx_dates_incoherentes = np.random.choice(indices_valides, size=min(n_dates_incoherentes, len(indices_valides)), replace=False)

for idx in idx_dates_incoherentes:
    entree = df_hospitalisations.loc[idx, "date_entree"]
    df_hospitalisations.loc[idx, "date_sortie"] = entree - timedelta(days=random.randint(1, 5))

rapport_anomalies["dates_incoherentes"] = len(idx_dates_incoherentes)
print(f"{len(idx_dates_incoherentes)} incohérences de dates injectées")

229 incohérences de dates injectées


In [12]:
# --- Âges invalides ---
mask_age_valide = df_patients["age"].notna()
indices_age_valides = df_patients[mask_age_valide].index

n_ages_invalides = int(len(df_patients) * TAUX_AGES_INVALIDES)
idx_ages_invalides = np.random.choice(indices_age_valides, size=min(n_ages_invalides, len(indices_age_valides)), replace=False)

for idx in idx_ages_invalides:
    df_patients.loc[idx, "age"] = random.choice([-5, -1, 150, 200])

rapport_anomalies["ages_invalides"] = len(idx_ages_invalides)
print(f"{len(idx_ages_invalides)} âges invalides injectés")

81 âges invalides injectés


## 8. Résumé des anomalies injectées

In [13]:
df_rapport_anomalies = pd.DataFrame(
    list(rapport_anomalies.items()), columns=["type_anomalie", "nb_lignes_affectees"]
)
df_rapport_anomalies

,type_anomalie,nb_lignes_affectees
0,valeurs_manquantes_hospitalisations,600
1,valeurs_manquantes_patients,320
2,doublons_hospitalisations,300
3,doublons_patients,160
4,dates_incoherentes,229
5,ages_invalides,81


## 9. Export des fichiers CSV bruts

In [14]:
df_patients.to_csv(f"{OUTPUT_DIR}/patients.csv", index=False)
df_services.to_csv(f"{OUTPUT_DIR}/services.csv", index=False)
df_medecins.to_csv(f"{OUTPUT_DIR}/medecins.csv", index=False)
df_hospitalisations.to_csv(f"{OUTPUT_DIR}/hospitalisations.csv", index=False)
df_rapport_anomalies.to_csv(f"{OUTPUT_DIR}/rapport_anomalies_injectees.csv", index=False)

print("Fichiers exportés dans data/raw/ ✅")
print(f"- patients.csv : {len(df_patients)} lignes")
print(f"- services.csv : {len(df_services)} lignes")
print(f"- medecins.csv : {len(df_medecins)} lignes")
print(f"- hospitalisations.csv : {len(df_hospitalisations)} lignes")

Fichiers exportés dans data/raw/ ✅
- patients.csv : 8160 lignes
- services.csv : 16 lignes
- medecins.csv : 103 lignes
- hospitalisations.csv : 15300 lignes


## 10. Vérification rapide (sanity check)

In [15]:
print("Aperçu patients :")
display(df_patients.info())

print("\nAperçu hospitalisations :")
display(df_hospitalisations.info())

print("\nNombre de médecins par service :")
display(df_medecins.groupby("specialite")["id_medecin"].count().sort_values(ascending=False))

print("\nRépartition des pathologies (top 10) :")
display(df_hospitalisations["pathologie"].value_counts().head(10))

print("\nRépartition des modes de sortie :")
display(df_hospitalisations["mode_sortie"].value_counts())

Aperçu patients :
<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_patient        8160 non-null   int64  
 1   num_dossier       8160 non-null   str    
 2   nom               8160 non-null   str    
 3   prenom            8160 non-null   str    
 4   date_naissance    8160 non-null   object 
 5   age               7996 non-null   float64
 6   sexe              8160 non-null   str    
 7   ville             8160 non-null   str    
 8   telephone         7995 non-null   str    
 9   groupe_sanguin    8160 non-null   str    
 10  couverture_sante  8160 non-null   str    
dtypes: float64(1), int64(1), object(1), str(8)
memory usage: 701.4+ KB


None


Aperçu hospitalisations :
<class 'pandas.DataFrame'>
RangeIndex: 15300 entries, 0 to 15299
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_hospitalisation  15300 non-null  int64  
 1   id_patient          15300 non-null  int64  
 2   id_service          15300 non-null  int64  
 3   id_medecin          15094 non-null  float64
 4   date_entree         15300 non-null  object 
 5   date_sortie         15097 non-null  object 
 6   type_admission      15300 non-null  str    
 7   pathologie          15098 non-null  str    
 8   mode_sortie         15300 non-null  str    
 9   numero_chambre      15300 non-null  str    
dtypes: float64(1), int64(3), object(2), str(4)
memory usage: 1.2+ MB


None


Nombre de médecins par service :


specialite
Cardiologie           8
Médecine interne      8
Orthopédie            8
Oncologie             8
Néphrologie           7
ORL                   7
Maternité             7
Chirurgie générale    7
Urologie              7
Réanimation           7
Neurologie            6
Pneumologie           6
Pédiatrie             5
Gastro-entérologie    5
Urgences              4
Dermatologie          3
Name: id_medecin, dtype: int64


Répartition des pathologies (top 10) :


pathologie
Sclérose en plaques            224
Occlusion intestinale          219
Méningite                      219
Anémie sévère                  217
Brûlure étendue                217
Hernie inguinale               214
Néphrite                       213
Eczéma sévère                  213
Rétention urinaire             212
Détresse respiratoire aiguë    207
Name: count, dtype: int64


Répartition des modes de sortie :


mode_sortie
Guéri                              8498
Amélioré                           3803
Transfert vers un autre service    1524
Sortie contre avis médical          863
Décès                               612
Name: count, dtype: int64

---
### Ce qui a été ajouté par rapport à la version précédente

- **Table `medecins`** : rattachés à un service, avec spécialité et années d'expérience
- **Services** : 9 → 16, catégories réalistes de CHU (dont Réanimation, ORL, Neurologie...)
- **Patients** : + numéro de dossier, téléphone, groupe sanguin, couverture santé
- **Hospitalisations** : + médecin référent, pathologie (cohérente avec le service), mode de sortie, numéro de chambre
- **Volumes** : 8000 patients / 15 000 hospitalisations (volume complet du cahier des charges)

### Prochaine étape
Ouvrir `02_audit_qualite_donnees.ipynb` pour auditer et nettoyer ces fichiers bruts (désormais 4 fichiers au lieu de 3).